In [16]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [18]:
def emnist_fix(x):
    x = torch.rot90(x, k=1, dims=[1, 2])
    x = torch.flip(x, dims=[2])
    return x

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(emnist_fix),
    transforms.Normalize((0.5,), (0.5,))
])


In [19]:
train_data = datasets.EMNIST(
    root="data", split="balanced", train=True, download=True, transform=transform
)

test_data = datasets.EMNIST(
    root="data", split="balanced", train=False, download=True, transform=transform
)

In [20]:
train_loader = DataLoader(
    train_data, batch_size=64, shuffle=True, num_workers=0, pin_memory=False
)
test_loader = DataLoader(
    test_data, batch_size=64, shuffle=False, num_workers=0, pin_memory=False
)

In [21]:
num_classes = len(train_data.classes)
print("Number of classes in EMNIST: ", num_classes)

Number of classes in EMNIST:  47


In [22]:
class EMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 47),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [23]:
model = EMNIST_CNN().to(device)

In [24]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [25]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            preds = model(images)
            loss = loss_fn(preds, labels)
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        epoch_loss /= len(train_loader)
            
        print(f"Epoch: {epoch} | Loss: {epoch_loss:.4f}")

In [26]:
train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs=10)

Epoch: 0 | Loss: 0.8551
Epoch: 1 | Loss: 0.5245
Epoch: 2 | Loss: 0.4665
Epoch: 3 | Loss: 0.4266
Epoch: 4 | Loss: 0.3987
Epoch: 5 | Loss: 0.3755
Epoch: 6 | Loss: 0.3535
Epoch: 7 | Loss: 0.3395
Epoch: 8 | Loss: 0.3249
Epoch: 9 | Loss: 0.3116


In [27]:
def test_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            preds = model(images)
            
            predicted = preds.argmax(dim=1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return correct / total

In [28]:
test_model(model, test_loader)

0.8872340425531915

In [29]:
import json

with open("emnist_classes.json", "w") as f:
    json.dump(train_data.classes, f)

In [30]:
torch.save(model.state_dict(), "emnist_model.pth")